# P2 - FigQuant TinyLlama verification (resumable)

This notebook stores per-layer checkpoints and final results in Google Drive. If Colab interrupts during a layer, rerun the final cell with `RESUME=True`; completed layers are skipped. The checkpoint is written atomically after every completed layer.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
P2_DIR = '/content/drive/MyDrive/littlefig-p2'
import os
os.makedirs(P2_DIR, exist_ok=True)
print('Durable P2 directory:', P2_DIR)

In [ ]:
!pip install -q torch transformers psutil numpy
import os, subprocess
REPO = '/content/littlefig'
BRANCH = 'research/p1-figmezo-verify'
if not os.path.exists(REPO + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--branch', BRANCH, 'https://github.com/Harboria-Labs/littlefig.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'switch', '-q', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'pull', '-q', '--ff-only'], check=True)
%cd /content/littlefig
print('Source checkout ready:', BRANCH)

In [ ]:
# Set this False only when intentionally starting a new checkpoint.
RESUME = True
MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
MEMORY_BUDGET_GIB = 8.0
CHECKPOINT_DIR = P2_DIR
RESULTS_PATH = P2_DIR + '/figquant_v2_results.json'
args = '--models ' + MODEL + ' --checkpoint-dir ' + CHECKPOINT_DIR + ' --results-path ' + RESULTS_PATH + ' --memory-budget-gib ' + str(MEMORY_BUDGET_GIB)
if RESUME: args += ' --resume'
print('Run:', 'python benchmark/experiment_figquant_v2.py ' + args)

In [ ]:
!python benchmark/experiment_figquant_v2.py {args}

In [ ]:
import json, os
checkpoint = os.path.join(P2_DIR, 'figquant_v2_checkpoint_TinyLlama_TinyLlama-1.1B-Chat-v1.0.json')
memory_status = os.path.join(P2_DIR, 'figquant_v2_memory_TinyLlama_TinyLlama-1.1B-Chat-v1.0.json')
print('Checkpoint:', checkpoint, os.path.exists(checkpoint))
if os.path.exists(checkpoint):
    d = json.load(open(checkpoint))
    print('Completed:', len(d.get('layers', {})), '/', d.get('total_layers'))
if os.path.exists(memory_status):
    print('Last durable memory heartbeat:')
    print(json.dumps(json.load(open(memory_status)), indent=2))
if os.path.exists(RESULTS_PATH): print(json.dumps(json.load(open(RESULTS_PATH)), indent=2))